# Physical moves: chess move -> arm primitive steps

**Deliverable:** physical move execution for every special move (README, "Known limitations":
*captures, castling, en passant, and promotion need their own sequences*).

A legal `chess.Move` on a `chess.Board` becomes an ordered list of arm primitives. The arm side
(`gambit_arm`) implements the primitives; this notebook only defines the protocol and the plans.

| Move | Steps |
|---|---|
| normal | `move_piece(from, to)` |
| capture | `capture_to_graveyard(to)`, `move_piece(from, to)` (clear the target first) |
| castling | `move_piece(king_from, king_to)`, `move_piece(rook_from, rook_to)` (king first, FIDE touch rule) |
| en passant | `capture_to_graveyard(captured_pawn_square)` (not the target square!), `move_piece(from, to)` |
| promotion | `capture_to_graveyard(from)` (pawn leaves), `place_from_graveyard(new_piece, to)` |
| capture + promotion | `capture_to_graveyard(to)`, `capture_to_graveyard(from)`, `place_from_graveyard(new_piece, to)` |

Promotion pieces come from the graveyard: the spare queens (`pieces.spare_queens` in
`gambit_bringup/config/board.yaml`) plus everything captured so far. `Graveyard` tracks that inventory.

**TODO checklist**
- [x] Plans for every move type, verified by replaying them on a piece map.
- [ ] Decide what to do when the promotion piece is not in the graveyard (e.g. underpromotion to
      a rook while both rooks are still on the board): ask the human, or use an upside-down rook
      as a queen... (`MissingPieceError` is raised for now).
- [ ] Decide graveyard slot layout (which slot holds which piece) together with `gambit_arm`.
- [ ] Optional: order steps to minimise arm travel (e.g. capture then move from the nearest side).

In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Protocol

import chess
import yaml

BOARD_CFG_PATH = Path("../../gambit_bringup/config/board.yaml")
with open(BOARD_CFG_PATH) as _f:
    BOARD_CFG = yaml.safe_load(_f)
SPARE_QUEENS = BOARD_CFG["pieces"]["spare_queens"]

## Arm interface (implemented by `gambit_arm`, faked in `game_loop.ipynb`)

Squares are python-chess square indices (`chess.E4`). Each primitive returns `True` on success
(the arm side does its own grasp verification and retries; see `recovery.max_arm_retries`).

In [ ]:
class ArmInterface(Protocol):
    def pick(self, square: chess.Square) -> bool: ...
    def place(self, square: chess.Square) -> bool: ...
    def move_piece(self, from_sq: chess.Square, to_sq: chess.Square) -> bool: ...
    def capture_to_graveyard(self, square: chess.Square) -> bool: ...
    def place_from_graveyard(self, piece: chess.Piece, square: chess.Square) -> bool: ...
    def park(self) -> bool: ...

## Steps

A `Step` is one primitive call. `piece` is the piece that moves (or, for `place_from_graveyard`,
the piece taken out of the graveyard); it is carried along so the arm can choose a grasp height
from `pieces.height` in board.yaml and so the plan can be checked offline.

In [ ]:
MOVE, CAPTURE, FROM_GRAVEYARD = "move_piece", "capture_to_graveyard", "place_from_graveyard"


@dataclass(frozen=True)
class Step:
    op: str
    square: chess.Square                  # source square (move/capture) or target (from graveyard)
    to: Optional[chess.Square] = None     # target square for move_piece
    piece: Optional[chess.Piece] = None

    def __str__(self) -> str:
        sym = self.piece.symbol() if self.piece else "?"
        if self.op == MOVE:
            return f"move_piece({sym} {chess.square_name(self.square)}->{chess.square_name(self.to)})"
        if self.op == CAPTURE:
            return f"capture_to_graveyard({sym} {chess.square_name(self.square)})"
        return f"place_from_graveyard({sym} -> {chess.square_name(self.square)})"


class MissingPieceError(RuntimeError):
    """The piece needed for a promotion is not in the graveyard."""

## Graveyard inventory

Starts with the spare queens; every capture adds the captured piece, every promotion removes the
promoted piece. (Pieces that left the board are the only ones the arm can bring back.)

In [ ]:
class Graveyard:
    def __init__(self, spare_queens: int = SPARE_QUEENS):
        self.pieces: Counter = Counter({chess.Piece(chess.QUEEN, chess.WHITE): spare_queens,
                                        chess.Piece(chess.QUEEN, chess.BLACK): spare_queens})

    def add(self, piece: chess.Piece) -> None:
        self.pieces[piece] += 1

    def take(self, piece: chess.Piece) -> None:
        if self.pieces[piece] <= 0:
            raise MissingPieceError(f"no {piece.symbol()} in the graveyard")
        self.pieces[piece] -= 1

    def has(self, piece: chess.Piece) -> bool:
        return self.pieces[piece] > 0

    def copy(self) -> "Graveyard":
        g = Graveyard(0)
        g.pieces = Counter(self.pieces)
        return g

## Planning

`plan_physical_move` reads the board *before* the move. python-chess encodes castling as the king
moving two squares (`e1g1`), so the rook squares are derived from the castling side. Chess960
castling (king "captures" own rook) is handled by `board.is_castling` the same way.

In [ ]:
def castling_rook_squares(board: chess.Board, move: chess.Move) -> tuple[chess.Square, chess.Square]:
    """(rook_from, rook_to) for a castling move."""
    rank = chess.square_rank(move.from_square)
    if board.is_kingside_castling(move):
        rook_to = chess.square(5, rank)                       # f-file
        rook_from = (move.to_square if board.chess960 else chess.square(7, rank))
    else:
        rook_to = chess.square(3, rank)                       # d-file
        rook_from = (move.to_square if board.chess960 else chess.square(0, rank))
    return rook_from, rook_to


def king_castle_target(board: chess.Board, move: chess.Move) -> chess.Square:
    rank = chess.square_rank(move.from_square)
    return chess.square(6 if board.is_kingside_castling(move) else 2, rank)


def plan_physical_move(board: chess.Board, move: chess.Move,
                       graveyard: Optional[Graveyard] = None) -> list[Step]:
    """Ordered arm steps for a legal move. Updates `graveyard` (if given) as the plan would."""
    if move not in board.legal_moves:
        raise ValueError(f"illegal move {move.uci()} in {board.fen()}")
    gy = graveyard.copy() if graveyard is not None else Graveyard()   # commit only on success
    mover = board.piece_at(move.from_square)
    steps: list[Step] = []

    if board.is_castling(move):
        rook_from, rook_to = castling_rook_squares(board, move)
        rook = board.piece_at(rook_from)
        steps.append(Step(MOVE, move.from_square, king_castle_target(board, move), mover))
        steps.append(Step(MOVE, rook_from, rook_to, rook))
        return steps                                   # castling never touches the graveyard

    if board.is_en_passant(move):
        captured_sq = chess.square(chess.square_file(move.to_square), chess.square_rank(move.from_square))
        captured = board.piece_at(captured_sq)
        steps.append(Step(CAPTURE, captured_sq, piece=captured))
        gy.add(captured)
        steps.append(Step(MOVE, move.from_square, move.to_square, mover))
        _commit(graveyard, gy)
        return steps

    if board.is_capture(move):
        captured = board.piece_at(move.to_square)
        steps.append(Step(CAPTURE, move.to_square, piece=captured))
        gy.add(captured)

    if move.promotion:
        new_piece = chess.Piece(move.promotion, mover.color)
        steps.append(Step(CAPTURE, move.from_square, piece=mover))     # pawn leaves the board
        gy.add(mover)
        gy.take(new_piece)                                             # may raise MissingPieceError
        steps.append(Step(FROM_GRAVEYARD, move.to_square, piece=new_piece))
    else:
        steps.append(Step(MOVE, move.from_square, move.to_square, mover))
    _commit(graveyard, gy)
    return steps


def _commit(target: Optional[Graveyard], planned: Graveyard) -> None:
    if target is not None:
        target.pieces = planned.pieces


def describe_plan(steps: list[Step]) -> str:
    return "; ".join(str(s) for s in steps)

## Executing and replaying plans

`execute_plan` sends the steps to an `ArmInterface` and stops at the first failure (the game
manager then re-reads the board and goes to recovery). `apply_steps` replays steps on a plain
`{square: piece}` dict; it is how the fake arm in `game_loop.ipynb` moves pieces and how the
self-check proves every plan produces the right position.

In [ ]:
@dataclass
class ExecutionResult:
    ok: bool
    steps_done: int
    failed_step: Optional[Step] = None


def execute_plan(arm: ArmInterface, steps: list[Step]) -> ExecutionResult:
    for i, s in enumerate(steps):
        if s.op == MOVE:
            ok = arm.move_piece(s.square, s.to)
        elif s.op == CAPTURE:
            ok = arm.capture_to_graveyard(s.square)
        elif s.op == FROM_GRAVEYARD:
            ok = arm.place_from_graveyard(s.piece, s.square)
        else:
            raise ValueError(s.op)
        if not ok:
            return ExecutionResult(False, i, s)
    return ExecutionResult(True, len(steps))


def apply_steps(piece_map: dict, steps: list[Step], graveyard: Optional[Counter] = None) -> dict:
    """Replay steps on a copy of a {square: Piece} map. Raises if a step is physically impossible."""
    pm = dict(piece_map)
    gy = graveyard if graveyard is not None else Counter()
    for s in steps:
        if s.op == MOVE:
            assert s.square in pm, f"{s}: source empty"
            assert s.to not in pm, f"{s}: target occupied"
            pm[s.to] = pm.pop(s.square)
        elif s.op == CAPTURE:
            assert s.square in pm, f"{s}: nothing to capture"
            gy[pm.pop(s.square)] += 1
        elif s.op == FROM_GRAVEYARD:
            assert s.square not in pm, f"{s}: target occupied"
            pm[s.square] = s.piece
    return pm

## Self-check

For each special move: plan it, replay the plan on the board's piece map and compare with the
position python-chess computes after the move.

In [ ]:
def _check(fen: str, uci: str, expected_ops: list[str], gy: Optional[Graveyard] = None) -> list[Step]:
    board = chess.Board(fen)
    move = chess.Move.from_uci(uci)
    steps = plan_physical_move(board, move, gy)
    assert [s.op for s in steps] == expected_ops, (uci, describe_plan(steps))
    after = apply_steps(board.piece_map(), steps)
    board.push(move)
    assert after == board.piece_map(), (uci, describe_plan(steps))
    return steps

START = chess.STARTING_FEN
_check(START, "e2e4", [MOVE])                                                        # normal
_check("rnbqkbnr/ppp1pppp/8/3p4/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2", "e4d5", [CAPTURE, MOVE])
s = _check("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1", "e1g1", [MOVE, MOVE])           # O-O
assert (s[0].square, s[0].to, s[1].square, s[1].to) == (chess.E1, chess.G1, chess.H1, chess.F1)
assert s[0].piece.piece_type == chess.KING                                          # king first
s = _check("r3k2r/8/8/8/8/8/8/R3K2R b KQkq - 0 1", "e8c8", [MOVE, MOVE])           # O-O-O black
assert (s[1].square, s[1].to) == (chess.A8, chess.D8)
s = _check("4k3/8/8/3pP3/8/8/8/4K3 w - d6 0 2", "e5d6", [CAPTURE, MOVE])           # en passant
assert s[0].square == chess.D5                                                       # not d6!
gy = Graveyard()
s = _check("8/4P3/8/8/8/8/k7/4K3 w - - 0 1", "e7e8q", [CAPTURE, FROM_GRAVEYARD], gy)  # promotion
assert gy.pieces[chess.Piece(chess.QUEEN, chess.WHITE)] == SPARE_QUEENS - 1
assert gy.pieces[chess.Piece(chess.PAWN, chess.WHITE)] == 1
_check("3r4/4P3/8/8/8/8/k7/4K3 w - - 0 1", "e7d8q", [CAPTURE, CAPTURE, FROM_GRAVEYARD])  # capture+promo
gy = Graveyard()
gy.add(chess.Piece(chess.ROOK, chess.WHITE))               # a white rook was captured earlier
_check("3r4/4P3/8/8/8/8/k7/4K3 w - - 0 1", "e7d8r", [CAPTURE, CAPTURE, FROM_GRAVEYARD], gy)
assert not gy.has(chess.Piece(chess.ROOK, chess.WHITE)) and gy.has(chess.Piece(chess.ROOK, chess.BLACK))
gy = Graveyard()
try:                                                       # underpromotion with no knight available
    plan_physical_move(chess.Board("8/4P3/8/8/8/8/k7/4K3 w - - 0 1"), chess.Move.from_uci("e7e8n"), gy)
    raise AssertionError("expected MissingPieceError")
except MissingPieceError:
    assert gy.pieces[chess.Piece(chess.PAWN, chess.WHITE)] == 0   # failed plan left inventory untouched

# replay a whole short game (with castling and captures) through the planner
board, gy, pm = chess.Board(), Graveyard(), chess.Board().piece_map()
for san in "e4 e5 Nf3 Nc6 Bc4 Nf6 O-O Nxe4 Re1 d5 Bxd5 Qxd5 Nc3 Qa5 Rxe4+ Be7".split():
    mv = board.parse_san(san)
    pm = apply_steps(pm, plan_physical_move(board, mv, gy))
    board.push(mv)
    assert pm == board.piece_map(), san
print("physical_moves self-check passed")